## 🎯 Learning Objectives
* Design and implement an agent capable of performing research and generating reports.
* Integrate multiple tools into an agent's decision-making process.
* Apply ReAct-like principles to orchestrate agent actions.
* Evaluate an agent's performance based on its output and internal reasoning.


# AG03-L08 Exercise: Build a Fully Functional Research-and-Report Agent

## Task Description

In this exercise, you will build a fully functional AI agent from scratch that can perform research on a given topic and then synthesize its findings into a concise report. This agent will demonstrate core capabilities such as tool utilization, reasoning, and iterative refinement, mimicking a simplified ReAct (Reasoning and Acting) loop.

Imagine you are tasked with creating an automated system to quickly gather and summarize information on emerging technological trends for a business intelligence unit. Your agent should be able to take a research query, use available tools to find information, and then compile that information into a structured report.

## Requirements

1.  **Agent Core**: Implement an `Agent` class or a set of functions that encapsulate the agent's logic, state, and interaction loop.
2.  **Tool Integration**: The agent must effectively utilize two provided mock tools:
    *   `search_tool(query: str) -> str`: Simulates a web search, returning relevant information.
    *   `report_writer_tool(title: str, content: str) -> str`: Simulates generating a formatted report.
3.  **Reasoning Loop**: The agent should exhibit a basic 


In [ ]:
import json
import time

# --- Mock LLM and Tools Setup ---

class MockLLM:
    """A mock Language Model that simulates reasoning and tool calls based on prompt keywords."""
    def __init__(self):
        self.call_count = 0
        self.search_results_cache = {}
        self.report_content_cache = {}

    def __call__(self, prompt: str) -> str:
        self.call_count += 1
        print(f"\n--- MockLLM Call {self.call_count} ---")
        print(f"Prompt: {prompt[:200]}...") # Print first 200 chars of prompt

        # Simple state machine / keyword matching for deterministic responses
        if "Final Answer:" in prompt:
            # If the agent has already decided on a final answer, just confirm
            return "Thought: The agent has already provided a final answer. No further action needed."

        if "Observation:" not in prompt and "search_tool" not in prompt:
            # Initial state: Need to search
            query_start = prompt.find("Query: ") + len("Query: ")
            query_end = prompt.find("\nHistory:")
            research_query = prompt[query_start:query_end].strip()
            return f"Thought: The user wants me to research '{research_query}'. I should start by searching for general information on this topic.\nAction: search_tool\nAction Input: \"{research_query}\""

        elif "search_tool" in prompt and "Observation:" in prompt and "search results for" in prompt:
            # After search, time to synthesize and write report
            # Extract the research query from the initial prompt to use in the report title
            query_start = prompt.find("Query: ") + len("Query: ")
            query_end = prompt.find("\nHistory:")
            research_query = prompt[query_start:query_end].strip()

            # Extract search observation
            obs_start = prompt.rfind("Observation: ") + len("Observation: ")
            observation = prompt[obs_start:].strip()

            # Simulate synthesizing content from observation
            synthesized_content = f"Based on the research query '{research_query}' and the search results: {observation[:150]}... The key findings indicate that... [further simulated synthesis]."
            self.report_content_cache[research_query] = synthesized_content

            report_title = f"Research Report on {research_query}"
            return f"Thought: I have gathered information. Now I need to synthesize it and write a report.\nAction: report_writer_tool\nAction Input: {json.dumps({'title': report_title, 'content': synthesized_content})}"

        elif "report_writer_tool" in prompt and "Observation:" in prompt and "Report generated successfully" in prompt:
            # After report generation, provide final answer
            # Retrieve the generated report content from cache
            query_start = prompt.find("Query: ") + len("Query: ")
            query_end = prompt.find("\nHistory:")
            research_query = prompt[query_start:query_end].strip()
            final_report_content = self.report_content_cache.get(research_query, "No report content found.")
            return f"Thought: I have successfully generated the report. Here is the final output.\nFinal Answer: {final_report_content}"

        return "Thought: I am unsure how to proceed. This is an unexpected state.\nAction: None\nAction Input: None"


def search_tool(query: str) -> str:
    """Simulates a web search and returns relevant information."""
    print(f"Calling search_tool with query: '{query}'")
    time.sleep(0.5) # Simulate network latency
    if "quantum computing" in query.lower() and "cryptography" in query.lower():
        return "Search results for 'impact of quantum computing on cryptography': Quantum computing, particularly Shor's algorithm, poses a significant threat to current public-key cryptography standards like RSA and ECC. Grover's algorithm could also weaken symmetric-key ciphers. Post-quantum cryptography (PQC) is an active research area developing new cryptographic algorithms resistant to quantum attacks. NIST has been standardizing PQC algorithms since 2016."
    elif "AI ethics" in query.lower():
        return "Search results for 'AI ethics': Key concerns in AI ethics include bias in algorithms, privacy violations, accountability for AI decisions, job displacement, and the potential for autonomous weapons. Frameworks like 'responsible AI' and 'trustworthy AI' are being developed by governments and organizations to address these issues."
    else:
        return f"Search results for '{query}': No specific detailed information found for this exact query, but general knowledge suggests [some generic info related to AI/tech]."


def report_writer_tool(title: str, content: str) -> str:
    """Simulates generating a formatted report."""
    print(f"Calling report_writer_tool with title: '{title}' and content (first 100 chars): '{content[:100]}...' ")
    time.sleep(0.3) # Simulate processing time
    formatted_report = f"""
# {title}

## Introduction

This report summarizes findings on the topic of '{title.replace('Research Report on ', '')}'.

## Key Findings

{content}

## Conclusion

Further research may be required to delve deeper into specific aspects.
"""
    return formatted_report


def parse_llm_output(output: str):
    """Parses the LLM's output to extract Thought, Action, Action Input, or Final Answer."""
    thought_match = "Thought: "
    action_match = "Action: "
    action_input_match = "Action Input: "
    final_answer_match = "Final Answer: "

    thought = ""
    action = None
    action_input = None
    final_answer = None

    lines = output.split('\n')
    for line in lines:
        if line.startswith(thought_match):
            thought = line[len(thought_match):].strip()
        elif line.startswith(action_match):
            action = line[len(action_match):].strip()
        elif line.startswith(action_input_match):
            action_input_str = line[len(action_input_match):].strip()
            try:
                # Attempt to parse as JSON for tool inputs
                action_input = json.loads(action_input_str)
            except json.JSONDecodeError:
                # If not JSON, treat as plain string
                action_input = action_input_str.strip('"') # Remove quotes if present
        elif line.startswith(final_answer_match):
            final_answer = line[len(final_answer_match):].strip()

    return {
        "thought": thought,
        "action": action,
        "action_input": action_input,
        "final_answer": final_answer
    }


# Define available tools for the agent
available_tools = {
    "search_tool": search_tool,
    "report_writer_tool": report_writer_tool
}

print("Setup complete: MockLLM and tools are ready.")


## Your Implementation

Now it's your turn to implement the `ResearchAgent` class. Your agent should:

1.  Take a `query` as input to its `run` method.
2.  Maintain a `history` or `scratchpad` to keep track of its thoughts, actions, and observations.
3.  Use the `MockLLM` to decide its next step (Thought, Action, Action Input, or Final Answer).
4.  Execute actions by calling the appropriate functions from the `available_tools` dictionary.
5.  Update its history with the observation from the tool execution.
6.  Continue this loop until the `MockLLM` indicates a `Final Answer`.
7.  Return the `Final Answer` (the generated report).

Feel free to structure your agent's internal logic as you see fit, but ensure it adheres to the requirements outlined above. Remember to make your code clean and well-commented.


In [ ]:
class ResearchAgent:
    """An AI agent capable of researching a topic and generating a report using tools."""
    def __init__(self, llm: MockLLM, tools: dict):
        self.llm = llm
        self.tools = tools
        self.history = [] # Stores (step_type, content) tuples

    def _build_prompt(self, query: str) -> str:
        """Constructs the prompt for the LLM based on the current query and history."""
        prompt_parts = [
            "You are a research agent. Your goal is to research a given topic and write a comprehensive report.",
            "You have access to the following tools: search_tool, report_writer_tool.",
            "Your output should follow the Thought/Action/Action Input or Thought/Final Answer format.",
            "Query: " + query,
            "History:"
        ]
        for step_type, content in self.history:
            prompt_parts.append(f"{step_type}: {content}")
        return "\n".join(prompt_parts)

    def run(self, query: str, max_steps: int = 10) -> str:
        """Executes the research and reporting process for a given query."""
        self.history = [] # Reset history for a new run
        print(f"\n--- Starting Research Agent for Query: '{query}' ---")

        for step in range(max_steps):
            print(f"\n--- Agent Step {step + 1} ---")
            current_prompt = self._build_prompt(query)
            llm_output = self.llm(current_prompt)
            parsed_output = parse_llm_output(llm_output)

            thought = parsed_output["thought"]
            action = parsed_output["action"]
            action_input = parsed_output["action_input"]
            final_answer = parsed_output["final_answer"]

            self.history.append(("Thought", thought))
            print(f"Agent Thought: {thought}")

            if final_answer:
                print(f"Agent decided to provide a Final Answer.")
                print(f"Final Report:\n{final_answer}")
                return final_answer

            if action and action_input is not None:
                print(f"Agent Action: {action}")
                print(f"Agent Action Input: {action_input}")

                if action in self.tools:
                    tool_func = self.tools[action]
                    try:
                        # Handle different tool input types (string vs. dict)
                        if isinstance(action_input, dict):
                            observation = tool_func(**action_input)
                        else:
                            observation = tool_func(action_input)
                        self.history.append(("Observation", observation))
                        print(f"Observation: {observation[:100]}...") # Print first 100 chars of observation
                    except Exception as e:
                        error_msg = f"Error executing tool {action}: {e}"
                        self.history.append(("Observation", error_msg))
                        print(f"Observation: {error_msg}")
                        # If a tool fails, the agent might need to re-evaluate or terminate
                        return f"Agent failed due to tool error: {error_msg}"
                else:
                    error_msg = f"Unknown tool: {action}"
                    self.history.append(("Observation", error_msg))
                    print(f"Observation: {error_msg}")
                    return f"Agent failed due to unknown tool: {error_msg}"
            else:
                print("Agent did not specify a valid action or final answer. Terminating.")
                return "Agent terminated without a clear action or final answer."

        print("Max steps reached. Agent terminated.")
        return "Agent terminated: Max steps reached without a final answer."

# --- Instantiate and Run the Agent ---

mock_llm_instance = MockLLM()
research_agent = ResearchAgent(llm=mock_llm_instance, tools=available_tools)

# Example 1: Research on Quantum Computing and Cryptography
query_1 = "the impact of quantum computing on cryptography"
final_report_1 = research_agent.run(query_1)

print("\n=====================================================")
print("Final Report for Query 1:")
print(final_report_1)
print("=====================================================\n")

# Example 2: Research on AI Ethics
query_2 = "key challenges in AI ethics"
final_report_2 = research_agent.run(query_2)

print("\n=====================================================")
print("Final Report for Query 2:")
print(final_report_2)
print("=====================================================\n")
